# HW2 - Neural Network from Scratch (PyTorch)

We build a 1-hidden-layer neural network (784 -> 200 -> 1) using only PyTorch tensors (no `nn.Linear`).

Architecture:
- Input layer: 784 neurons (28x28 pixels)
- Hidden layer: 200 neurons, ReLU activation
- Output layer: 1 neuron, Sigmoid activation

In [1]:
import torch
import pandas as pd

torch.manual_seed(0)

## Load data

Each row = [label, 784 pixel values]. We split into X (pixels) and y (label), then normalize pixels from [0, 255] to [0, 1].

In [2]:
train_df = pd.read_csv("mnist_train_binary.csv")
test_df = pd.read_csv("mnist_test_binary.csv")

# first column is the label, rest are pixels
X_train = torch.tensor(train_df.iloc[:, 1:].values, dtype=torch.float32) / 255.0
y_train = torch.tensor(train_df.iloc[:, 0].values, dtype=torch.float32).reshape(-1, 1)

X_test = torch.tensor(test_df.iloc[:, 1:].values, dtype=torch.float32) / 255.0
y_test = torch.tensor(test_df.iloc[:, 0].values, dtype=torch.float32).reshape(-1, 1)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: torch.Size([11339, 784])
y_train shape: torch.Size([11339, 1])


## 1. Build Neural Network (from scratch)

We manually define weights and activation functions instead of using `nn.Linear`.

Following the diagram's notation, weight matrix `W[l]` has shape `(n_out, n_in)`, so a layer computes `a[l] = activation(W[l] @ a[l-1])`.

- Xavier init (used for the Sigmoid output layer): `sqrt(2) / sqrt(n_in + n_out)`
- He init (used for the ReLU hidden layer): `sqrt(2) * sqrt(2) / sqrt(n_in + n_out)`

In [3]:
def relu(z):
    return torch.clamp(z, min=0)

def sigmoid(z):
    return 1 / (1 + torch.exp(-z))

In [4]:
n_input = 784
n_hidden = 200
n_output = 1

torch.manual_seed(0)

# He initialization for the ReLU hidden layer, W1 shape: (n_hidden, n_input)
he_std = torch.sqrt(torch.tensor(2.0)) * torch.sqrt(torch.tensor(2.0)) / torch.sqrt(torch.tensor(float(n_input + n_hidden)))
W1 = torch.randn(n_hidden, n_input) * he_std

# Xavier initialization for the Sigmoid output layer, W2 shape: (n_output, n_hidden)
xavier_std = torch.sqrt(torch.tensor(2.0)) / torch.sqrt(torch.tensor(float(n_hidden + n_output)))
W2 = torch.randn(n_output, n_hidden) * xavier_std

def forward(X):
    a1 = relu(X @ W1.T)   # hidden layer, shape (samples, n_hidden)
    a2 = sigmoid(a1 @ W2.T)  # output layer, shape (samples, n_output)
    return a2

## 2. Calculate loss & accuracy

Using the first 64 samples of the train set.

In [5]:
X_batch = X_train[:64]
y_batch = y_train[:64]

y_hat = forward(X_batch)

### 2-1. Binary cross entropy loss

In [6]:
m = y_batch.shape[0]
loss = -(1 / m) * torch.sum(y_batch * torch.log(y_hat) + (1 - y_batch) * torch.log(1 - y_hat))
print("Loss:", loss.item())

Loss: 0.6674728393554688


### 2-2. Accuracy

In [7]:
predictions = (y_hat >= 0.5).float()
accuracy = torch.sum(predictions == y_batch).item() / m
print("Accuracy:", accuracy)

Accuracy: 0.65625
